In [10]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine,URL

from google.cloud import bigquery
from google.oauth2 import service_account

In [11]:
credentials = service_account.Credentials.from_service_account_file(
  'c:/Users/Bob/oasisbiz/datawarehouse-390004-34bcb00fb7cb.json'
)
project_id = 'datawarehouse-390004'

In [12]:
client = bigquery.Client(
  project=project_id,
  credentials=credentials
)

In [2]:
# connect to DB
conn = psycopg2.connect(
  host='localhost',
  port=5432,
  dbname='postgres',
  user='postgres',
  password='postgres'
)
conn.set_session(autocommit=True)
cursor = conn.cursor()

In [3]:
engine = create_engine(
  URL.create(
    drivername='postgresql+psycopg2',
    host='localhost',
    port=5432,
    database='postgres',
    username='postgres',
    password='postgres'
  )
)

In [25]:
cursor.execute(
  f'''
  select
    sales_amt,
    base_dt,
    extract('year' from base_dt) base_year,
    extract('month' from base_dt) base_month,
    market_name name
  from market_sales
  where
    base_dt >= '2021-01-01' and
    base_dt < '2025-01-01'
  '''
)
y_df = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

---
X 구성

In [ ]:
# 상권 기하학적 속성
cursor.execute(
  f'''
  select
    cname name,
    round(st_perimeter(geom_3857)) perimeter,
    round(st_area(geom_3857)) area,
    st_perimeter(geom_3857) / st_area(geom_3857) linear_value,
    st_area(geom_3857) / st_area(st_convexhull(geom_3857)) convex_value,
    st_area(geom_3857) / st_area(st_minimumboundingcircle(geom_3857)) circular_value
  from market_polygon
  '''
)
mk_geom_attr = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [15]:
# 매장 특성
job = client.query(
  f'''
  select
    store.base_dt,
    market.site_name name,
    count(store_no) store_cnt,
    round(sum(
      case
        when category_first_name = '외식' then 1
        else 0
      end
    )) store_restaurant_cnt,
    round(sum(
      case
        when category_first_name = '서비스' then 1
        else 0
      end
    )) store_service_cnt,
    round(sum(
      case
        when category_first_name = '도소매' then 1
        else 0
      end
    )) store_retail_cnt,
    round(sum(
      case
        when category_first_name = '기타' then 1
        else 0
      end
    )) store_etc_cnt
  from m2.sh_bldg_sales store,
  (
  select
    site_name,
    site_area,
    geometry geom
  from biz.market_site_2024
  where sd_code = '11'
  ) as market
  where
    store.base_dt >= '2021-01-01' and
    store.base_dt < '2025-01-01' and
    st_intersects(
      market.geom,
      store.point
    )
  group by 1,2
'''
)
mk_store = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [16]:
# 거래 특성
job = client.query(
  f'''
  select
    store.base_dt,
    market.site_name name,
    sum(est_cnt) sales_cnt,
    round(sum(
      case
        when category_first_name = '외식' then est_cnt
        else 0
      end
    )) sales_restaurant_cnt,
    round(sum(
      case
        when category_first_name = '서비스' then est_cnt
        else 0
      end
    )) sales_service_cnt,
    round(sum(
      case
        when category_first_name = '도소매' then est_cnt
        else 0
      end
    )) sales_retail_cnt,
    round(sum(
      case
        when category_first_name = '기타' then est_cnt
        else 0
      end
    )) sales_etc_cnt,
    round(sum(est_cnt*wk_rt/100)) sales_wk_cnt,
    round(sum(est_cnt*we_rt/100)) sales_we_cnt,
    round(sum(est_cnt*time_0510_rt/100)) sales_morning_cnt,
    round(sum(est_cnt*(time_1114_rt+time_1517_rt)/100)) sales_afternoon_cnt,
    round(sum(est_cnt*(time_1819_rt+time_2021_rt)/100)) sales_evening_cnt,
    round(sum(est_cnt*(time_2224_rt+time_0104_rt)/100)) sales_night_cnt,
    round(sum(est_cnt*(m20_rt+m30_rt+m40_rt+m50_rt+m60_rt)/100)) sales_male_cnt,
    round(sum(est_cnt*(f20_rt+f30_rt+f40_rt+f50_rt+f60_rt)/100)) sales_female_cnt,
    round(sum(est_cnt*(f20_rt+m20_rt)/100)) sales_age20_cnt,
    round(sum(est_cnt*(f30_rt+m30_rt)/100)) sales_age30_cnt,
    round(sum(est_cnt*(f40_rt+m40_rt)/100)) sales_age40_cnt,
    round(sum(est_cnt*(f50_rt+m50_rt)/100)) sales_age50_cnt,
    round(sum(est_cnt*(f60_rt+m60_rt)/100)) sales_age60_cnt
  from m2.sh_bldg_sales store,
  (
  select
    site_name,
    site_area,
    geometry geom
  from biz.market_site_2024
  where sd_code = '11'
  ) as market
  where
    store.base_dt >= '2021-01-01' and
    store.base_dt < '2025-01-01' and
    st_intersects(
      market.geom,
      store.point
    )
  group by 1,2
'''
)
mk_sales = job.result().to_dataframe()

---
m3_y + x 통합

In [26]:
m3_total_df = y_df.merge(
  mk_geom_attr,
  how='left',
  on='name'
).merge(
  mk_store,
  how='left',
  on=['base_dt','name']
).merge(
  mk_sales,
  how='left',
  on=['base_dt','name']
)

In [28]:
m3_total_df.to_sql(
  'm3_total',
  engine,
  if_exists='replace',
  index=False
)

486

---
전달용 데이터 저장하기

In [29]:
m3_total_df.to_csv(
  'm3_data_06.22.csv',
  sep=',',
  index=False
)